# Study 937 — Tranches 🍰

**"Rebalanced monthly" hides a coin-flip: *which* day of the month?**

Take an ordinary sleeve — hold **SPY** for the next month if it closed above its **200-day**
average on the rebalance day, otherwise hold **IEF**. Nothing exotic. But "monthly" is not
one schedule, it is **21** of them: the book rebalanced on the 3rd trading day of the cycle
is a different book from the one rebalanced on the 10th, running the *identical rule*.

Study [836](../../836-timing-luck/) showed on synthetic data that this choice alone moves
the reported Sharpe. This study asks the follow-up on the **real tape**: how big is the
gap, and does the textbook fix — **tranching**, splitting the book into N sleeves
rebalanced on N staggered dates — actually close it, and at what price?

Real tape: SPY x IEF, 2002-07-30 → 2026-06-30, books live from 2003-06-16
(5,797 days), excess-of-cash both sides, 5 bps one-way. Lag: the state known at
the close of *t* is the position that earns the return of *t+1* (one shift).

*Every real number below is frozen from `docs/results.md` (Fingerprint `e8f9552be0ac`,
as-of 2026-06-30). The live cells at the end run the offline **synthetic** control and are
labelled as such.*


## 1. The same rule, twenty-one answers

Run the sleeve on each of the 21 possible rebalance days and you get 21 track records. They are not close together. Over 23 years the luckiest calendar choice finished **95% richer** than the unluckiest — same rule, same assets, same costs, same everything except the day of the month.

In [1]:
R = dict(n1_min=0.499, n1_max=0.755, n1_range=0.256, n1_tw=95.1, n1_cagr_sd=0.86)
print('worst rebalance date : excess Sharpe %+.3f' % R['n1_min'])
print('best  rebalance date : excess Sharpe %+.3f' % R['n1_max'])
print('gap from luck alone  : %.3f Sharpe, %.1f%% of terminal wealth'
      % (R['n1_range'], R['n1_tw']))
print('year-in-year-out spread in CAGR: %.2f pp (standard deviation)'
      % R['n1_cagr_sd'])

worst rebalance date : excess Sharpe +0.499
best  rebalance date : excess Sharpe +0.755
gap from luck alone  : 0.256 Sharpe, 95.1% of terminal wealth
year-in-year-out spread in CAGR: 0.86 pp (standard deviation)


## 2. It is luck, not skill — a rule with **no** edge has a wider gap

If the spread came from some dates being genuinely better at timing markets, a rule with no timing ability would show no spread. It shows more: an exposure-matched **random-timing** rule on the same two funds disperses **0.462** of a Sharpe point across the 21 dates and **419%** of terminal wealth. One random schedule is itself a coin-flip, so we ran **20** of them: the edge-free cone is wider than the trend rule's in **19 of 20** (mean sd 0.093 against 0.066). The cone is an accident of the sampling schedule, and every monthly strategy on earth has one.

## 3. The fix: stop betting the book on one date

Split the capital into **N sleeves**, each rebalanced on a different day of the cycle, each left to compound on its own. Nobody has to guess the lucky date, because you own all of them. Here is the spread of outcomes as N grows:

In [2]:
rows = [(1, 0.066, 95.1, 2.741), (4, 0.023, 27.3, 2.71), (12, 0.01, 7.7, 2.7), (21, 0.0, 0.0, 2.7)]
print('tranches   Sharpe spread (sd)   terminal-wealth spread   traded/yr')
for n, sd, tw, turn in rows:
    print('%8d   %17.3f   %20.1f%%   %9.2fx' % (n, sd, tw, turn))
print('\nfour sleeves already remove two-thirds of the luck; twenty-one remove all of it,')
print('and the book trades LESS, not more.')

tranches   Sharpe spread (sd)   terminal-wealth spread   traded/yr
       1               0.066                   95.1%        2.74x
       4               0.023                   27.3%        2.71x
      12               0.010                    7.7%        2.70x
      21               0.000                    0.0%        2.70x

four sleeves already remove two-thirds of the luck; twenty-one remove all of it,
and the book trades LESS, not more.


> 🔬 **For the quants.** The collapse is faster than the naive 1/√N: sd(N)/sd(1) = 1.000 / 0.340 / 0.152 / 0.000 against 1.000 / 0.500 / 0.289 / 0.218. At N = 21 it is *exactly* zero — every rotation of a 21-tranche book is the same set of 21 offsets, so there is nothing left to be lucky about. It is arithmetic, not an estimate.

## 4. What it costs — and what it is worth

Turnover barely moves (**2.74x → 2.70x** of NAV a year, -1.5%), because each sleeve only trades 1/N of the book. The risk-adjusted return improves a little. But be honest about *where* that improvement comes from:

In [3]:
print('one date (average of 21) -> 21 tranches')
print('  excess Sharpe  %+.3f  ->  %+.3f   (%+.3f)' % (0.631, 0.676, 0.044))
print('  CAGR           %5.2f%%  ->  %5.2f%%   (%+.2f pp/yr)' % (9.63, 9.7, 0.07))
print('  volatility     %5.2f%%  ->  %5.2f%%' % (13.3, 12.32))
print('  worst drawdown %5.1f%%  ->  %5.1f%%' % (-35.4, -26.2))
print()
print('Almost all of the Sharpe gain is LOWER RISK, not higher return:')
print('seven basis points a year of extra growth, one full point less volatility.')

one date (average of 21) -> 21 tranches
  excess Sharpe  +0.631  ->  +0.676   (+0.044)
  CAGR            9.63%  ->   9.70%   (+0.07 pp/yr)
  volatility     13.30%  ->  12.32%
  worst drawdown -35.4%  ->  -26.2%

Almost all of the Sharpe gain is LOWER RISK, not higher return:
seven basis points a year of extra growth, one full point less volatility.


## 5. The bill nobody prints: broker tickets

Percentage costs do not care how many sleeves you run — each trades 1/N of the book. **Tickets do.** A switch in 21 sleeves is 21 pairs of orders instead of one: **2.6 → 57.6 tickets a year**. A ticket costs a fixed sum, so what it costs *you* depends entirely on how big your account is — at 0.5 bp of NAV per ticket the full 21-sleeve book pays **0.29%/yr**, which is four times the 0.07 pp/yr of growth it added. Inside a large fund it costs nothing. This is an **assumption**, not something we can read off the price tape — which is exactly why the sensible retail answer is **four sleeves** (11 tickets/yr, 0.055%/yr) and not twenty-one.

## 6. Could you just pick the lucky date instead?

Maybe once. Split the history in half: the two halves' rankings of the 21 dates correlate **ρ = +0.309**. The first half's winner did rank 2/21 in the second half — worth **+0.058** over the average date, which on those same rows was actually *more* than the **+0.042** tranching gave over the same stretch. We say so plainly rather than compare it against the flattering full-sample number. But that is **one draw**, on a date chosen by looking at the first half, and the second half's actual winner was a different date (offset 9) — so the chaser is right back in the lottery, betting the whole book on a day of the month. Two different rules do tend to like the same dates (ρ = +0.556), which hints at a month-end flow footprint underneath; the fix needs no forecast at all.

## 7. Live check — the machinery is unbiased (offline synthetic)

On a synthetic world with a **planted** regime the sleeve genuinely beats a matched random control; on a **null** world it does not. In *both* worlds the timing-luck cone is there and tranching kills it — which is the point: the fix is schedule arithmetic, it neither needs edge nor creates any.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from tranching import data, strategy as st
import numpy as np
for tag, ss in [('planted regime', 1.0), ('flat null     ', 0.0)]:
    rows = [st.synthetic_detect(p) for p, _ in
            data.synthetic_panel(3, signal_strength=ss, n_years=40)]
    edge = np.array([d['sleeve_minus_random'] for d in rows])
    sd1 = np.mean([d['sharpe_sd_n1'] for d in rows])
    sdf = np.mean([d['sharpe_sd_full'] for d in rows])
    print('%s (3 worlds): sleeve vs random control %+.3f  |  cone sd %.3f -> %.3f'
          % (tag, edge.mean(), sd1, sdf))

planted regime (3 worlds): sleeve vs random control +0.536  |  cone sd 0.047 -> 0.000


flat null      (3 worlds): sleeve vs random control +0.044  |  cone sd 0.052 -> 0.000


## Verdict

- **Signal — Real.** The cone is a fact of the real tape, not a simulation artefact: **0.256** of a Sharpe point and **95%** of terminal wealth decided by the calendar, sd **0.066**, and the same picture in both eras, on a second rule, on a second cash leg, with an extra day of execution delay, and wider on a rule with no edge at all. (It is a *measurement*, not a bet won: a spread has no null to reject, and a standard deviation cannot straddle zero, so we lean on size and replication rather than on a p-value.) Tranching closes it exactly, for **-1.5%** turnover and **+0.044** of Sharpe.
- **Tradability — Fragile.** What you bank is **0.07 pp/yr** and a quieter ride, not alpha — and the ticket bill (an assumption, not tape) can eat it whole on a small account. Tranch so your track record stops being an accident of the calendar; four sleeves buy most of that, twenty-one buy the rest at a price only a large book can ignore.